# Conditional Motif-Scaffolding

Now, we'll do the same, but instead of generating a structure from scratch, we will give RFdiffusion an existing protein and ask it to generate a missing part of it (similar to image in-painting).

In [1]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import py3Dmol
from matplotlib.colors import ListedColormap

from scripts.utils import (
    count_amino_acids, load_reversed_trajectory, get_pdb_plddt,
    PLDDT_CONFIDENCE_RANGES, get_plddt_ranges, align_structures,
    load_trb, get_motif_hal_idx0, contiguous_runs,
)

ModuleNotFoundError: No module named 'py3Dmol'

In [2]:
out_dir = Path("out_motif_scaffolding")
out_dir.mkdir(exist_ok=True)

### Structure generation with RFdiffusion

In [ ]:
out_dir_rfdiffusion = out_dir / "out_rfdiffusion"
out_dir_rfdiffusion.mkdir(parents=True, exist_ok=True)

Activate environment:
`conda activate protein-design`

Run RFdiffusion motif-scaffolding inference in the terminal. We scaffold RSV-F site V (residues 163-181 of chain A in `5TPN.pdb`, one of the motif-scaffolding benchmark examples shipped with RFdiffusion), asking the model to build 10-40 residues on each side of the fixed motif:

Alternatively, use a different protein: download your favourite protein from `rcsb.org` and select a motif of ~20 amino acids to scaffold

```
MDLDIR=./data/generative_models
python ${MDLDIR}/RFdiffusion/scripts/run_inference.py \
    inference.output_prefix=out_motif_scaffolding/out_rfdiffusion/design \
    inference.input_pdb=${MDLDIR}/RFdiffusion/examples/input_pdbs/5TPN.pdb \
    'contigmap.contigs=[10-40/A163-181/10-40]' \
    inference.num_designs=10
```

Wait until it finishes generating the 10 designs, then run the following cells to visualize the designed structures. Unlike unconditional design, RFdiffusion keeps the motif's backbone (and residue identity) fixed and only builds the surrounding scaffold.

In [ ]:
n_designs = 10
n_cols = 5
n_rows = n_designs // n_cols

pdb_paths = [out_dir_rfdiffusion / f"design_{i}.pdb" for i in range(n_designs)]
trb_paths = [out_dir_rfdiffusion / f"design_{i}.trb" for i in range(n_designs)]
missing = [str(p) for p in pdb_paths + trb_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing output files:\n" + "\n".join(missing))

view = py3Dmol.view(
    width=200*n_cols, height=200*n_rows,
    viewergrid=(n_rows, n_cols), linked=False,
)

for i, (pdb_path, trb_path) in enumerate(zip(pdb_paths, trb_paths)):
    r, c = divmod(i, n_cols)
    cell = (r, c)

    pdb_text = pdb_path.read_text(encoding="utf-8")
    num_aa = count_amino_acids(pdb_text)
    design_motif_resi = (get_motif_hal_idx0(load_trb(trb_path)) + 1).tolist()

    view.addModel(pdb_text, "pdb", viewer=cell)
    view.setViewStyle({"style": "outline"}, viewer=cell)
    view.setStyle({"cartoon": {"color": "lightgray"}}, viewer=cell)
    view.addStyle(
        {"resi": design_motif_resi},
        {"cartoon": {"color": "orange"}, "stick": {"radius": 0.15, "color": "orange"}},
        viewer=cell,
    )
    view.addLabel(
        f"design_{i} | {num_aa} aa | motif {len(design_motif_resi)} aa",
        {"position": {"x": 0, "y": 0, "z": 0}, "backgroundColor": "white", "inFront": True,
         "fontColor": "black", "fontSize": 13, "alignment": "topCenter",
         "screenOffset": {"x": 0, "y": 100}},
        viewer=cell,
    )
    view.zoomTo({"resi": design_motif_resi}, viewer=cell)
    view.zoom(0.6, viewer=cell)

view.show()

Pick one design and show the diffusion trajectory. The motif (orange) stays fixed while the surrounding scaffold (gray/spectrum) denoises around it.

In [ ]:
n_design = 0  # from 0 to 9

In [ ]:
motif_trb = load_trb(out_dir_rfdiffusion / f"design_{n_design}.trb")
motif_idx0 = get_motif_hal_idx0(motif_trb)
motif_resi = (motif_idx0 + 1).tolist()

print(f"Motif: {len(motif_idx0)} fixed residues at output positions {motif_resi}")

In [ ]:
xt_path = out_dir_rfdiffusion / "traj" / f"design_{n_design}_Xt-1_traj.pdb"
px0_path = out_dir_rfdiffusion / "traj" / f"design_{n_design}_pX0_traj.pdb"

xt_text, xt_n_frames = load_reversed_trajectory(xt_path, hold_final_frames=10)
px0_text, px0_n_frames = load_reversed_trajectory(px0_path, hold_final_frames=10)

if xt_n_frames != px0_n_frames:
    print(f"Warning: Xt-1 has {xt_n_frames} frames, pX0 has {px0_n_frames} frames")

trajectory_viewer = py3Dmol.view(width=1000, height=400, viewergrid=(1, 2), linked=True)
panels = [("Predicted X0", px0_text, (0, 0), 0.8), ("Xt-1", xt_text, (0, 1), 0.8)]

for title, traj_text, cell, zoom in panels:
    trajectory_viewer.addModelsAsFrames(traj_text, "pdb", viewer=cell)
    # Full cartoon
    trajectory_viewer.setStyle({"cartoon": {"color": "spectrum", "opacity": 0.7}}, viewer=cell)
    # CA-only beads
    trajectory_viewer.addStyle(
        {"atom": "CA"}, {"sphere": {"radius": 0.45, "color": "spectrum"}}, viewer=cell,
    )
    # Backbone sticks
    trajectory_viewer.addStyle(
        {"atom": ["N", "CA", "C"]}, {"stick": {"radius": 0.12, "color": "spectrum"}}, viewer=cell,
    )
    # Fixed motif, highlighted
    trajectory_viewer.addStyle(
        {"resi": motif_resi}, {"stick": {"radius": 0.2, "color": "orange"}}, viewer=cell,
    )
    trajectory_viewer.addLabel(
        title,
        {"position": {"x": 0, "y": 0, "z": 0}, "backgroundColor": "white", "inFront": True,
         "fontColor": "black", "fontSize": 16, "screenOffset": {"x": -50, "y": 200}},
        viewer=cell,
    )
    trajectory_viewer.zoomTo({"resi": motif_resi}, viewer=cell)
    trajectory_viewer.zoom(zoom, viewer=cell)

trajectory_viewer.animate({"loop": "forward", "interval": 300})
trajectory_viewer.show()

### Structure-conditioned sequence generation (inverse folding) with ProteinMPNN
As before, generate possible sequences for the generated structure (notice that Protein MPNN is also constrained to match the sequence of the known input motif). 

In [ ]:
out_dir_mpnn = out_dir / "out_mpnn"
out_dir_mpnn.mkdir(parents=True, exist_ok=True)

In [ ]:
fixed_positions = {}
for i in range(n_designs):
    design_trb = load_trb(out_dir_rfdiffusion / f"design_{i}.trb")
    fixed_positions[f"design_{i}"] = {"A": (get_motif_hal_idx0(design_trb) + 1).tolist()}

fixed_positions_path = out_dir_mpnn / "fixed_positions.jsonl"
with open(fixed_positions_path, "w", encoding="utf-8") as f:
    f.write(json.dumps(fixed_positions) + "\n")

Activate environment:
`conda activate protein-design`

Run ProteinMPNN in the terminal for your chosen design. `--fixed_positions_jsonl` keeps the motif residues (site V) at their native identity, so only the scaffold that RFdiffusion built around it gets redesigned:

```
N=0
MDLDIR=./data/generative_models
python ${MDLDIR}/ProteinMPNN/protein_mpnn_run.py \
    --pdb_path out_motif_scaffolding/out_rfdiffusion/design_${N}.pdb \
    --pdb_path_chains A \
    --fixed_positions_jsonl out_motif_scaffolding/out_mpnn/fixed_positions.jsonl \
    --out_folder out_motif_scaffolding/out_mpnn \
    --path_to_model_weights ${MDLDIR}/ProteinMPNN/vanilla_model_weights \
    --num_seq_per_target 5 \
    --sampling_temp 0.1
```

Wait until it finishes generating the 5 sequences, then run the following cells to analyze the generated sequences

In [ ]:
mpnn_fasta = out_dir_mpnn / "seqs" / f"design_{n_design}.fa"

mpnn_records = []
header = None
sequence = []
for line in mpnn_fasta.read_text(encoding="utf-8").splitlines():
    if line.startswith(">"):
        if header is not None:
            mpnn_records.append((header, "".join(sequence)))
        header = line[1:]
        sequence = []
    else:
        sequence.append(line.strip())
if header is not None:
    mpnn_records.append((header, "".join(sequence)))

labels = ["backbone"] + [f"sample {i}" for i in range(1, len(mpnn_records))]
aligned_sequences = [sequence for _, sequence in mpnn_records]
alphabet = sorted(set("".join(aligned_sequences)))
aa_to_index = {aa: i for i, aa in enumerate(alphabet)}
alignment_matrix = np.array([[aa_to_index[aa] for aa in sequence] for sequence in aligned_sequences])

fig, ax = plt.subplots(figsize=(12, 0.4 * len(aligned_sequences) + 1.5))
for start, end in contiguous_runs(motif_idx0):
    ax.axvspan(start - 0.5, end + 0.5, color="orange", alpha=0.15, zorder=0)
ax.imshow(alignment_matrix, cmap=ListedColormap(plt.cm.tab20.colors[:len(alphabet)]), aspect="auto")
for row, sequence in enumerate(aligned_sequences):
    for column, aa in enumerate(sequence):
        ax.text(column, row, aa, ha="center", va="center", fontsize=9)
ax.set_xticks(range(len(aligned_sequences[0])))
ax.set_xticklabels([])
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel("Residue position")
ax.set_title("ProteinMPNN generated sequences (shaded = fixed motif)")
ax.tick_params(length=0)
plt.tight_layout()
plt.show()

In [ ]:
with open(out_dir_mpnn / f"mpnn_design_{n_design}.fasta", "w", encoding="utf-8") as f:
    for i in range(1, len(aligned_sequences)):
        f.write(f">mpnn_{i}\n")
        f.write(aligned_sequences[i] + "\n")

### Structure prediction (folding) with ESMFold
Compare the structure generated by RFdiffusion and the predicted one based on the generated sequence, as before. Are there any differences?

In [ ]:
out_dir_esmfold = out_dir / "out_esmfold"
out_dir_esmfold.mkdir(parents=True, exist_ok=True)

Activate environment:
`conda activate esmfold`

Run ESMFold structure prediction of the 5 ProteinMPNN-generated sequences:

```
MDLDIR=./data/generative_models
HF_HOME=${MDLDIR}/ESMFold/hf-cache \
    python ${MDLDIR}/ESMFold/esmfold.py \
        --fastas_folder out_motif_scaffolding/out_mpnn \
        --output_folder out_motif_scaffolding/out_esmfold
```

In [ ]:
esmfold_pdbs = sorted(out_dir_esmfold.rglob("*.pdb"))
if not esmfold_pdbs:
    raise FileNotFoundError(f"Run ESMFold first; no PDB files found under {out_dir_esmfold}")

mpnn_sequences = aligned_sequences[1:]
if len(esmfold_pdbs) != len(mpnn_sequences):
    print(
        f"Warning: found {len(esmfold_pdbs)} ESMFold PDBs but {len(mpnn_sequences)} ProteinMPNN sequences."
    )

plddt_by_model = []
for i, pdb_path in enumerate(esmfold_pdbs):
    model_plddt = np.asarray(get_pdb_plddt(pdb_path), dtype=float)
    sequence = mpnn_sequences[i]
    if len(sequence) != len(model_plddt):
        raise ValueError(
            f"{pdb_path.name}: sequence length {len(sequence)} does not match pLDDT length {len(model_plddt)}"
        )
    plddt_by_model.append({
        "sample": i + 1,
        "pdb_path": pdb_path,
        "sequence": sequence,
        "plddt": model_plddt,
        "mean_plddt": float(np.mean(model_plddt)),
    })

best_model = max(plddt_by_model, key=lambda item: item["mean_plddt"])
mpnn_pdb_path = best_model["pdb_path"]
esmfold_sequence = best_model["sequence"]
plddt = best_model["plddt"]

print(f"Selected for downstream visualization: sample {best_model['sample']}")
print(f"PDB: {mpnn_pdb_path}")
print(f"Mean pLDDT: {best_model['mean_plddt']:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for plddt_range in PLDDT_CONFIDENCE_RANGES:
    ax.axhspan(
        plddt_range["lower"], plddt_range["upper"],
        color=plddt_range["color"], alpha=0.18,
    )
for start, end in contiguous_runs(motif_idx0):
    ax.axvspan(start + 0.5, end + 1.5, color="orange", alpha=0.12, zorder=0)

line_colors = [
    "#0072B2",  # blue
    "#D55E00",  # vermillion
    "#009E73",  # green
    "#CC79A7",  # purple
    "#E69F00",  # orange
]
for color, model in zip(line_colors, plddt_by_model):
    positions = np.arange(1, len(model["plddt"]) + 1)
    label = f"{model['sample']}) pLDDT={model['mean_plddt']:.1f}"
    ax.plot(
        positions, model["plddt"], marker="o", markersize=3,
        linewidth=2.5, color=color, label=label,
    )

max_len = len(model["plddt"])
x_ticks = np.unique(np.r_[1, np.arange(10, max_len + 1, 10), max_len])
ax.set_xlim(1, max_len)
ax.set_xticks(x_ticks)
ax.set_xticklabels([str(int(tick)) for tick in x_ticks])

ax.set(xlabel="Residue position", ylabel="pLDDT", ylim=(0, 100),
       title="ESMFold confidence for ProteinMPNN sequences (shaded = fixed motif)")
ax.grid(axis="y", alpha=0.2)

range_axis = ax.twinx()
range_axis.set_ylim(ax.get_ylim())
range_axis.set_yticks([
    (plddt_range["lower"] + plddt_range["upper"]) / 2
    for plddt_range in PLDDT_CONFIDENCE_RANGES
])
range_axis.set_yticklabels([plddt_range["label"] for plddt_range in PLDDT_CONFIDENCE_RANGES])
range_axis.tick_params(axis="y", length=0, pad=8)

ax.legend(
    bbox_to_anchor=(0.55, -0.3), loc="lower center", ncol=5,
    frameon=False, columnspacing=0.8, labelspacing=0.25,
)
plt.tight_layout()
plt.show()


In [ ]:
viewer = py3Dmol.view(width=600, height=400)
viewer.addModel(mpnn_pdb_path.read_text(encoding="utf-8"), "pdb")
viewer.setStyle({}, {"cartoon": {"color": "#0053d6"}})
plddt_ranges = get_plddt_ranges(plddt)
for residue_mask, color in plddt_ranges:
    residues = (np.flatnonzero(residue_mask) + 1).tolist()
    if not residues:
        continue
    selection = {"resi": residues}
    viewer.setStyle(selection, {"cartoon": {"color": color}})
    viewer.addStyle(selection, {"stick": {"radius": 0.1, "color": color}})
viewer.addStyle({"resi": motif_resi}, {"stick": {"radius": 0.18, "color": "orange"}})
viewer.zoomTo()
viewer.show()

In [ ]:
rf_pdb_path = Path(out_dir_rfdiffusion) / f"design_{n_design}.pdb"
rf_pdb_text = rf_pdb_path.read_text(encoding="utf-8")
esmfold_pdb_text = mpnn_pdb_path.read_text(encoding="utf-8")

alignment = align_structures(esmfold_pdb_text, rf_pdb_text)
aligned_esmfold_pdb = alignment["aligned_esmfold_pdb"]
ca_distances = alignment["ca_distances"]
ca_rmsd = alignment["ca_rmsd"]
tm_score = alignment["tm_score"]
length = alignment["length"]

motif_ca_rmsd = float(np.sqrt(np.mean(ca_distances[motif_idx0] ** 2)))

print(f"Aligned Ca residues: {length}")
print(f"Ca RMSD: {ca_rmsd:.3f} A")
print(f"TM-score: {tm_score:.3f}")
print(f"Motif Ca RMSD ({len(motif_idx0)} aa): {motif_ca_rmsd:.3f} A")

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(1, length + 1), ca_distances, color="teal", width=0.8)
for start, end in contiguous_runs(motif_idx0):
    ax.axvspan(start + 0.5, end + 1.5, color="orange", alpha=0.15, zorder=0)
ax.axhline(2.0, color="darkorange", linestyle="--", linewidth=1.5)
if len(esmfold_sequence) != length:
    raise ValueError(f"Sequence length {len(esmfold_sequence)} does not match aligned length {length}")
ax.set_xticks(range(1, length + 1))
ax.set_xticklabels(list(esmfold_sequence), fontfamily="monospace")
ax.set_xlim(0.5, length + 0.5)
ax.margins(x=0)
ax.set(xlabel="Amino acid sequence", ylabel="Ca RMSD (A)",
       title=f"RFdiffusion-ESMFold alignment deviation (shaded = motif, motif RMSD {motif_ca_rmsd:.2f} A)")
plt.tight_layout()
plt.show()

In [ ]:
view = py3Dmol.view(width=900, height=500)
view.addModel(rf_pdb_text, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "lightgray", "opacity": 0.8}})
view.setStyle({"model": 0, "resi": motif_resi}, {"cartoon": {"color": "orange", "opacity": 0.8}})
view.addStyle({"model": 0, "resi": motif_resi}, {"stick": {"radius": 0.12, "color": "orange"}})

view.addModel(aligned_esmfold_pdb, "pdb")
view.setStyle({"model": 1}, {"cartoon": {"color": "#0053d6"}})
plddt_ranges = get_plddt_ranges(plddt)
for residue_mask, color in plddt_ranges:
    residues = (np.flatnonzero(residue_mask) + 1).tolist()
    if not residues:
        continue
    selection = {"model": 1, "resi": residues}
    view.setStyle(selection, {"cartoon": {"color": color}})
    view.addStyle(selection, {"stick": {"radius": 0.1, "color": color}})
view.addStyle({"model": 1, "resi": motif_resi}, {"stick": {"radius": 0.12, "color": "orange"}})

label_style = {"backgroundColor": "white", "fontColor": "black", "inFront": True, "fontSize": 14}
view.addLabel("RFdiffusion: gray | ESMFold: pLDDT | motif: orange",
              {**label_style, "screenOffset": {"x": 200, "y": 18}})
view.addLabel(f"RMSD {ca_rmsd:.2f} A | TM-score {tm_score:.2f}",
              {**label_style, "screenOffset": {"x": 200, "y": -6}})
view.addLabel(f"Motif RMSD: {motif_ca_rmsd:.2f} A",
              {**label_style, "screenOffset": {"x": 200, "y": -30}})
view.zoomTo()
view.show()